In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/drive/MyDrive/etl-data-quality-monitoring/raw_sales_data.csv', encoding='latin1')
print("Shape:", df.shape)
df.head()

Mounted at /content/drive
Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [ ]:
print("Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)

Shape: (541909, 8)

Column Names: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data Types:
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object


In [ ]:
null_counts = df.isnull().sum()
null_percent = (df.isnull().sum() / len(df)) * 100

null_report = pd.DataFrame({
    'Null Count': null_counts,
    'Null %': round(null_percent, 2)
})
print("=== NULL REPORT ===")
print(null_report)

=== NULL REPORT ===
             Null Count  Null %
InvoiceNo             0    0.00
StockCode             0    0.00
Description        1454    0.27
Quantity              0    0.00
InvoiceDate           0    0.00
UnitPrice             0    0.00
CustomerID       135080   24.93
Country               0    0.00


In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Total Duplicate Records: {duplicate_count}")
print(f"Duplicate Rate: {round((duplicate_count / len(df)) * 100, 2)}%")

Total Duplicate Records: 5268
Duplicate Rate: 0.97%


In [ ]:
print("=== QUANTITY STATS ===")
print(df['Quantity'].describe())

print("\n=== UNIT PRICE STATS ===")
print(df['UnitPrice'].describe())

print("\n=== NEGATIVE QUANTITY (Returns/Cancellations) ===")
negative_qty = df[df['Quantity'] < 0]
print(f"Total Negative Quantity Records: {len(negative_qty)}")

print("\n=== ZERO UNIT PRICE ===")
zero_price = df[df['UnitPrice'] == 0]
print(f"Total Zero Price Records: {len(zero_price)}")

=== QUANTITY STATS ===
count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

=== UNIT PRICE STATS ===
count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

=== NEGATIVE QUANTITY (Returns/Cancellations) ===
Total Negative Quantity Records: 10624

=== ZERO UNIT PRICE ===
Total Zero Price Records: 2515


In [ ]:
df_cleaned = df.copy()

# 1. Remove Duplicates
df_cleaned = df_cleaned.drop_duplicates()
print(f"✅ Duplicates Removed: {df.shape[0] - df_cleaned.shape[0]}")

# 2. Fill Null Description with 'Unknown'
df_cleaned['Description'] = df_cleaned['Description'].fillna('Unknown')
print(f"✅ Null Descriptions Filled")

# 3. Drop rows where CustomerID is null
df_cleaned = df_cleaned.dropna(subset=['CustomerID'])
print(f"✅ Null CustomerID Rows Dropped")

# 4. Remove Negative Quantity (returns/cancellations)
df_cleaned = df_cleaned[df_cleaned['Quantity'] > 0]
print(f"✅ Negative Quantity Records Removed")

# 5. Remove Zero or Negative Unit Price
df_cleaned = df_cleaned[df_cleaned['UnitPrice'] > 0]
print(f"✅ Zero/Negative Unit Price Records Removed")

# 6. Fix Date Column
df_cleaned['InvoiceDate'] = pd.to_datetime(df_cleaned['InvoiceDate'])
print(f"✅ InvoiceDate Converted to DateTime")

# 7. Add TotalPrice column
df_cleaned['TotalPrice'] = df_cleaned['Quantity'] * df_cleaned['UnitPrice']
print(f"✅ TotalPrice Column Added")

print("\nCleaned Shape:", df_cleaned.shape)

✅ Duplicates Removed: 5268
✅ Null Descriptions Filled
✅ Null CustomerID Rows Dropped
✅ Negative Quantity Records Removed
✅ Zero/Negative Unit Price Records Removed
✅ InvoiceDate Converted to DateTime
✅ TotalPrice Column Added

Cleaned Shape: (392692, 9)


In [ ]:
print("=== FINAL NULL CHECK ===")
print(df_cleaned.isnull().sum())

print("\n=== FINAL DUPLICATE CHECK ===")
print(f"Duplicates: {df_cleaned.duplicated().sum()}")

print("\n=== FINAL NEGATIVE VALUE CHECK ===")
print(f"Negative Quantity: {len(df_cleaned[df_cleaned['Quantity'] < 0])}")
print(f"Negative UnitPrice: {len(df_cleaned[df_cleaned['UnitPrice'] < 0])}")

print("\n=== CLEANED DATASET SHAPE ===")
print(df_cleaned.shape)

=== FINAL NULL CHECK ===
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

=== FINAL DUPLICATE CHECK ===
Duplicates: 0

=== FINAL NEGATIVE VALUE CHECK ===
Negative Quantity: 0
Negative UnitPrice: 0

=== CLEANED DATASET SHAPE ===
(392692, 9)


In [ ]:
kpi = {
    'Total Records': len(df_cleaned),
    'Total Revenue': round(df_cleaned['TotalPrice'].sum(), 2),
    'Avg Order Value': round(df_cleaned['TotalPrice'].mean(), 2),
    'Unique Customers': df_cleaned['CustomerID'].nunique(),
    'Unique Products': df_cleaned['StockCode'].nunique(),
    'Total Countries': df_cleaned['Country'].nunique()
}

print("=== KPI SUMMARY ===")
for k, v in kpi.items():
    print(f"{k}: {v}")

=== KPI SUMMARY ===
Total Records: 392692
Total Revenue: 8887208.89
Avg Order Value: 22.63
Unique Customers: 4338
Unique Products: 3665
Total Countries: 37


In [ ]:
df_cleaned.to_csv('/content/drive/MyDrive/etl-data-quality-monitoring/cleaned_sales_data.csv', index=False)
print("✅ Cleaned data exported to Google Drive successfully!")

✅ Cleaned data exported to Google Drive successfully!


In [ ]:
import sqlite3
import pandas as pd

# Load cleaned data
df_cleaned = pd.read_csv('/content/drive/MyDrive/etl-data-quality-monitoring/cleaned_sales_data.csv')

# Create SQLite database
conn = sqlite3.connect('sales_data.db')

# Load dataframe into SQL table
df_cleaned.to_sql('sales_data', conn, if_exists='replace', index=False)
print("✅ Data loaded into SQLite database!")

✅ Data loaded into SQLite database!


In [ ]:
def run_query(query):
    result = pd.read_sql_query(query, conn)
    return result

In [ ]:
run_query("SELECT COUNT(*) AS total_records FROM sales_data")

,total_records
0,392692


In [ ]:
run_query("""
SELECT
    SUM(CASE WHEN InvoiceNo IS NULL THEN 1 ELSE 0 END) AS null_invoice,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer,
    SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description,
    SUM(CASE WHEN UnitPrice IS NULL THEN 1 ELSE 0 END) AS null_price
FROM sales_data
""")

,null_invoice,null_customer,null_description,null_price
0,0,0,0,0


In [ ]:
run_query("""
SELECT InvoiceNo, StockCode, CustomerID, COUNT(*) AS duplicate_count
FROM sales_data
GROUP BY InvoiceNo, StockCode, CustomerID
HAVING COUNT(*) > 1
LIMIT 10
""")

,InvoiceNo,StockCode,CustomerID,duplicate_count
0,536381,71270,15311.0,2
1,536409,85116,17908.0,2
2,536409,90199C,17908.0,3
3,536412,21448,17920.0,2
4,536412,21738,17920.0,3
5,536412,22077,17920.0,3
6,536412,22144,17920.0,2
7,536412,22243,17920.0,3
8,536412,22273,17920.0,2
9,536412,22749,17920.0,2


In [ ]:
run_query("""
SELECT
    COUNT(*) AS total_orders,
    ROUND(SUM(TotalPrice), 2) AS total_revenue,
    ROUND(AVG(TotalPrice), 2) AS avg_order_value,
    COUNT(DISTINCT CustomerID) AS unique_customers,
    COUNT(DISTINCT Country) AS total_countries
FROM sales_data
""")

,total_orders,total_revenue,avg_order_value,unique_customers,total_countries
0,392692,8887208.89,22.63,4338,37


In [ ]:
run_query("""
SELECT Country,
    ROUND(SUM(TotalPrice), 2) AS total_revenue
FROM sales_data
GROUP BY Country
ORDER BY total_revenue DESC
LIMIT 10
""")

,Country,total_revenue
0,United Kingdom,7285024.64
1,Netherlands,285446.34
2,EIRE,265262.46
3,Germany,228678.40
4,France,208934.31
5,Australia,138453.81
6,Spain,61558.56
7,Switzerland,56443.95
8,Belgium,41196.34
9,Sweden,38367.83


In [ ]:
run_query("""
SELECT
    strftime('%Y-%m', InvoiceDate) AS month,
    ROUND(SUM(TotalPrice), 2) AS monthly_revenue
FROM sales_data
GROUP BY month
ORDER BY month
""")

,month,monthly_revenue
0,2010-12,570422.73
1,2011-01,568101.31
2,2011-02,446084.92
3,2011-03,594081.76
4,2011-04,468374.33
5,2011-05,677355.15
6,2011-06,660046.05
7,2011-07,598962.90
8,2011-08,644051.04
9,2011-09,950690.20


In [ ]:
sql_code = """
-- ================================================
-- ETL Data Quality Monitoring - SQL Queries
-- Dataset: E-Commerce Sales Data
-- ================================================

-- 1. Total Records
SELECT COUNT(*) AS total_records
FROM sales_data;

-- 2. Null Value Check
SELECT
    SUM(CASE WHEN InvoiceNo IS NULL THEN 1 ELSE 0 END) AS null_invoice,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer,
    SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description,
    SUM(CASE WHEN UnitPrice IS NULL THEN 1 ELSE 0 END) AS null_price
FROM sales_data;

-- 3. Duplicate Records Check
SELECT InvoiceNo, StockCode, CustomerID, COUNT(*) AS duplicate_count
FROM sales_data
GROUP BY InvoiceNo, StockCode, CustomerID
HAVING COUNT(*) > 1;

-- 4. Negative Quantity Check (Returns)
SELECT COUNT(*) AS negative_quantity_records
FROM sales_data
WHERE Quantity < 0;

-- 5. Zero or Negative Price Check
SELECT COUNT(*) AS invalid_price_records
FROM sales_data
WHERE UnitPrice <= 0;

-- 6. KPI Summary
SELECT
    COUNT(*) AS total_orders,
    ROUND(SUM(TotalPrice), 2) AS total_revenue,
    ROUND(AVG(TotalPrice), 2) AS avg_order_value,
    COUNT(DISTINCT CustomerID) AS unique_customers,
    COUNT(DISTINCT Country) AS total_countries
FROM sales_data;

-- 7. Revenue by Country (Top 10)
SELECT Country,
    ROUND(SUM(TotalPrice), 2) AS total_revenue
FROM sales_data
GROUP BY Country
ORDER BY total_revenue DESC
LIMIT 10;

-- 8. Monthly Revenue Trend
SELECT
    strftime('%Y-%m', InvoiceDate) AS month,
    ROUND(SUM(TotalPrice), 2) AS monthly_revenue
FROM sales_data
GROUP BY month
ORDER BY month;
"""



In [ ]:

# saved to google drive
with open('/content/drive/MyDrive/etl-data-quality-monitoring/data_quality_checks.sql', 'w') as f:
    f.write(sql_code)
print("✅ SQL file saved to Google Drive!")

✅ SQL file saved to Google Drive!
